# Startup/Product Forecasting

Generate a forecasting dataset about startup survival using YC company data from [yc-oss/api](https://github.com/yc-oss/api). Questions focus on longevity (will this company still exist in X years?) and funding (will they reach Series A/B?). WebSearchLabeler verifies outcomes via web search. Seeds are stratified by outcome (Inactive vs Acquired/Public/Active) to reduce positive bias.

In [10]:
%pip install python-dotenv pandas
%pip install -e ..

from IPython.display import clear_output
clear_output()

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [11]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## YC company data

Fetch companies from [yc-oss/api](https://github.com/yc-oss/api). Filter by `launched_at` (2015–2022) so questions like "Will X still be around in 3 years?" have resolution dates in the past. Stratify by `status` (Inactive vs Acquired/Public/Active) for ~50/50 split to reduce positive bias.

In [12]:
import random
from datetime import datetime
from urllib.request import urlopen
import json

from lightningrod import create_sample

YC_API_URL = "https://yc-oss.github.io/api/companies/all.json"
# prefer older batches: they have more resolved outcomes
LAUNCH_START = int(datetime(2015, 1, 1).timestamp())
LAUNCH_END = int(datetime(2022, 1, 1).timestamp())
MIN_DESC_LENGTH = 100  # skip companies with little context for question generation
SAMPLES_PER_STATUS_GROUP = 175  # stratified sampling: ~50/50 failed vs successful to reduce positive bias

with urlopen(YC_API_URL) as resp:
    companies = json.load(resp)

def _build_seed_text(c):
    # exclude status: it would leak the outcome (Inactive vs Acquired) and trivialize the forecasting task
    desc = (c.get("long_description") or "").strip() or (c.get("one_liner") or "")
    tags = ", ".join(c.get("tags") or [])
    return (
        f"Title: {c.get('name', '')}\n"
        f"One-liner: {c.get('one_liner', '')}\n\n"
        f"Description: {desc}\n\n"
        f"Website: {c.get('website', '')}\n"
        f"YC URL: {c.get('url', '')}\n"
        f"Batch: {c.get('batch', '')}\n"
        f"Industry: {c.get('industry', '')}\n"
        f"Tags: {tags}"
    )

# launched_at range: older batches have more resolved outcomes; MIN_DESC_LENGTH ensures usable context
filtered = [
    c for c in companies
    if (c.get("launched_at") and LAUNCH_START <= c["launched_at"] < LAUNCH_END
        and len((c.get("long_description") or "") or (c.get("one_liner") or "")) >= MIN_DESC_LENGTH)
]

# stratify by outcome so input has ~50/50 failed vs successful; raw YC data skews toward success
inactive = [c for c in filtered if c.get("status") == "Inactive"]
successful = [c for c in filtered if c.get("status") in ("Acquired", "Public", "Active")]

random.seed(42)
n_inactive = min(len(inactive), SAMPLES_PER_STATUS_GROUP)
n_successful = min(len(successful), SAMPLES_PER_STATUS_GROUP)
sampled_inactive = random.sample(inactive, n_inactive)
sampled_successful = random.sample(successful, n_successful)
selected = sampled_inactive + sampled_successful
random.shuffle(selected)

samples = []
for c in selected:
    seed_text = _build_seed_text(c)
    seed_date = datetime.utcfromtimestamp(c["launched_at"]) if c.get("launched_at") else None
    samples.append(create_sample(seed_text, seed_date=seed_date))

input_dataset = lr.datasets.create_from_samples(samples, batch_size=1000)
print(f"Created input dataset: {input_dataset.id} ({len(samples)} companies: {n_inactive} Inactive, {n_successful} Acquired/Public/Active)")

Created input dataset: 14e5694c-6ae6-468a-b7f7-cefad5e7c21b (350 companies: 175 Inactive, 175 Acquired/Public/Active)


## Build the pipeline

ForwardLookingQuestionGenerator produces questions with prediction_date and date_close. NewsContextGenerator adds relevant news articles so the model has more than just the question text. WebSearchLabeler verifies survival via web search.

In [13]:
INSTRUCTIONS = """
Generate binary forecasting questions about whether this YC-backed startup will survive or succeed.
Focus on: (1) longevity — will the company still exist in X years? (2) funding — will they reach the next stage?
Use the company name, description, website, and YC URL to identify the startup. Questions must be forward-looking from the company's launch date and verifiable via web search.
"""

EXAMPLES = [
    "Will this company still be operational in 3 years?",
    "Will this startup still be around in 2 years?",
    "Will this company raise Series A within 18 months?",
]

BAD_EXAMPLES = [
    "What technology does this use?",
    "When was this founded?",
    "Is this B2B or B2C?",
]

In [14]:
from lightningrod import (
    BinaryAnswerType,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionRenderer,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    question_generator=ForwardLookingQuestionGenerator(
        instructions=INSTRUCTIONS,
        examples=EXAMPLES,
        bad_examples=BAD_EXAMPLES,
        answer_type=answer_type,
        questions_per_seed=2,
    ),
    context_generators=[
        NewsContextGenerator(
            num_search_queries=3,
            articles_per_query=3,
            num_articles=6,
            time_delta_days=60,  # news from ~2 months around prediction_date
        )
    ],
    labeler=WebSearchLabeler(
        answer_type=answer_type,
        confidence_threshold=0.5,
    ),
    renderer=QuestionRenderer(answer_type=answer_type),
)

> Note: Processing can take several minutes (question generation, news context fetch, web search labeling).

## Run the pipeline

In [ ]:
dataset = lr.transforms.run(
    pipeline,
    input_dataset=input_dataset,  # YC seeds from create_from_samples; no seed_generator
    max_questions=500,
    name="Startup forecasting",
)
samples = dataset.download()

pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

## Prepare the dataset

Filter valid samples, deduplicate, and split into train/test. Use `drop_missing_context=True` to keep only samples with news context. `days_to_resolution_range=(365, None)` keeps 1+ year horizons.

In [16]:
from lightningrod.training import prepare_for_training

train, test = prepare_for_training(
    samples,
    answer_type,
    test_size=0.2,
    split_strategy="temporal",
    include_assistant=True,
    filter_leaky_train=False,
    days_to_resolution_range=(365, None),  # keep only questions with 1+ year horizon (longevity/funding)
    drop_missing_context=True,  # require news context; question text alone isn't sufficient for prediction
)

for name, data in [("Train", train), ("Test", test)]:
    if data:
        yes_count = sum(s["label"] or 0 for s in data)
        print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")
    else:
        print(f"{name}: 0 rows")

Train: 66 rows, 33.3% yes
Test: 17 rows, 41.2% yes


## Results

In [17]:
def _display_head(data, name, n=5):
    if not data:
        print(f"{name}: no rows")
        return
    df = pd.DataFrame(data[:n])
    # Prevent question_text truncation in pandas DataFrame display
    pd.set_option('display.max_colwidth', None)
    cols = ["question_text", "prediction_date", "date_close", "resolution_date", "label", "label_confidence"]
    display_cols = [c for c in cols if c in df.columns]
    print(f"{name} (head):")
    display(df[display_cols])

_display_head(train, "Train")
_display_head(test, "Test")

Train (head):


,question_text,prediction_date,date_close,resolution_date,label,label_confidence
0,"Will Start Closing (startclosing.com) raise a Series A round of venture capital funding by December 31, 2018?",2015-11-13T17:20:16,2018-12-31T00:00:00,2018-12-31T00:00:00,0.0,1.0
1,"Will Start Closing (startclosing.com) still be an active, operational business entity on January 1, 2021?",2015-11-13T17:20:16,2021-01-01T00:00:00,2020-09-02T00:00:00,0.0,0.9
2,"Will Anchor Health (anchorhealth.com) still be an active company with an operational website on January 1, 2026?",2015-11-16T05:50:15,2026-01-01T00:00:00,2026-01-01T00:00:00,1.0,0.9
3,"Will Robby Technologies still be an active, independent operating business on April 21, 2021?",2016-04-21T17:40:18,2021-04-21T00:00:00,2021-04-21T00:00:00,1.0,0.8
4,"Will Robby Technologies raise a Series A funding round of at least $5 million on or before December 31, 2018?",2016-04-21T17:40:18,2018-12-31T00:00:00,2018-12-31T00:00:00,0.0,1.0


Test (head):


,question_text,prediction_date,date_close,resolution_date,label,label_confidence
0,"Will Covie announce a Series A funding round or a venture capital round of $10 million or more by December 31, 2023?",2021-06-24T01:40:44,2023-12-31T00:00:00,2023-12-31T00:00:00,0.0,1.0
1,"As of July 1, 2024, is the HVAC software company Onsite Pro (https://onsitepro.co) still an active and operational business entity?",2021-06-29T14:36:04,2024-07-01T00:00:00,2024-07-01T00:00:00,0.0,0.9
2,"Will Crew (crew.work) remain an active and operational business entity as of June 29, 2024?",2021-06-29T16:08:39,2024-06-29T00:00:00,2024-05-02T00:00:00,1.0,0.9
3,"Will Crew (crew.work) successfully raise a Series A funding round by December 31, 2023?",2021-06-29T16:08:39,2023-12-31T00:00:00,2023-12-31T00:00:00,0.0,1.0
4,"Will Dots (usedots.com) announce a Series B funding round by January 1, 2026?",2021-06-29T16:53:56,2026-01-01T00:00:00,2026-01-01T00:00:00,0.0,1.0


## Uploading the dataset to HuggingFace

Once we have a training-ready dataset, we can push it to Hugging Face for sharing or downstream use.

In [18]:
%pip install datasets -q

from datasets import Dataset, DatasetDict
from lightningrod.utils import config

dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test": Dataset.from_list(test),
})
print(f"Train: {len(dataset['train'])} rows, Test: {len(dataset['test'])} rows")
print("Columns:", dataset["train"].column_names[:8], "...")

DATASET_PATH = f"{config.get_config_value('HF_USERNAME')}/startup-forecasting-demo"
dataset.push_to_hub(DATASET_PATH, token=config.get_config_value("HF_ACCESS_TOKEN"))


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Train: 66 rows, Test: 17 rows
Columns: ['question_text', 'date_close', 'event_date', 'resolution_criteria', 'prediction_date', 'label', 'answer_type', 'label_confidence'] ...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/bart/startup-forecasting-demo/commit/781306735469ea85852723f8bdc1ce9d31ee467d', commit_message='Upload dataset', commit_description='', oid='781306735469ea85852723f8bdc1ce9d31ee467d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bart/startup-forecasting-demo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bart/startup-forecasting-demo'), pr_revision=None, pr_num=None)